# Vergleich: Annotation vs. CellSAM-Basismodell vs. feingetuntes Modell

1. Pfade in der **Konfiguration** anpassen
2. Modelle **einmal** laden (dauert)
3. In der letzten Zelle Bilder auswaehlen und anzeigen - beliebig oft neu ausfuehren

In [ ]:
import os
import random
import argparse
import numpy as np
import pandas as pd
import tifffile as tiff
from bioio import BioImage
from tqdm import tqdm

from cellSAM.sam_inference import CellSAM
import torch

In [ ]:
def read_image_any(path):
    """
    Returns a 2D numpy array (Y, X).
    - .tif/.tiff: reads as-is (if it’s already 2D MIP)
    - .czi: loads and max-projects over Z to produce a 2D MIP for the chosen (T, C, Scene)
    """
    ext = os.path.splitext(path)[1].lower()

    if ext in [".tif", ".tiff"]:
        arr = tiff.imread(path)
        arr = np.squeeze(arr)
        # If someone accidentally gives you (Z,Y,X), you can still MIP:
        if arr.ndim == 3:
            arr = arr.max(axis=0)
        if arr.ndim != 2:
            raise ValueError(f"TIFF not 2D after squeeze/MIP. shape={arr.shape} path={path}")
        return arr

    if ext == ".czi":
        
        img = BioImage(path)
        data = img.data

        d = np.asarray(data)

        if d.ndim == 5:      # T C Z Y X
            zyx = d[0, 0]
        elif d.ndim == 4:    # C Z Y X  or  T Z Y X
            zyx = d[0]
        elif d.ndim == 3:    # Z Y X
            zyx = d
        else:
            raise ValueError(f"Unsupported CZI array ndim={d.ndim}, shape={d.shape}, path={path}")

        mip = zyx.max(axis=0)  # max over Z -> (Y,X)
        return mip.astype(np.float32, copy=False)

    raise ValueError(f"Unsupported extension: {ext} for {path}")

def cxcywh_to_xyxy(boxes):
    
    # boxes: (N,4) normalized cxcywh -> normalized xyxy
    cx, cy, w, h = boxes[:,0], boxes[:,1], boxes[:,2], boxes[:,3]
    x1 = cx - 0.5*w
    y1 = cy - 0.5*h
    x2 = cx + 0.5*w
    y2 = cy + 0.5*h
    xyxy = np.stack([x1,y1,x2,y2], axis=1)
    
    return np.clip(xyxy, 0.0, 1.0)

def filter_border_boxes(boxes_xyxy: np.ndarray, H: int, W: int, margin: int = 0) -> np.ndarray:
    """
    Remove boxes that touch the image border (within margin pixels).
    boxes_xyxy are in pixel coordinates.
    """
    if boxes_xyxy is None or len(boxes_xyxy) == 0:
        return boxes_xyxy

    b = boxes_xyxy.astype(np.float32, copy=True)

    # clip to image bounds first (protect against tiny numerical overshoot)
    b[:, 0] = np.clip(b[:, 0], 0, W - 1)
    b[:, 2] = np.clip(b[:, 2], 0, W - 1)
    b[:, 1] = np.clip(b[:, 1], 0, H - 1)
    b[:, 3] = np.clip(b[:, 3], 0, H - 1)

    x1, y1, x2, y2 = b[:, 0], b[:, 1], b[:, 2], b[:, 3]

    # keep only boxes fully inside the margin
    keep = (x1 > margin) & (y1 > margin) & (x2 < (W - 1 - margin)) & (y2 < (H - 1 - margin))
    return b[keep]

def mip_to_uint8_rgb(img2d: np.ndarray, p_low=1.0, p_high=99.0) -> torch.Tensor:
    
    x = img2d.astype(np.float32)
    lo = np.percentile(x, p_low)
    hi = np.percentile(x, p_high)
    x = np.clip((x - lo) / (hi - lo + 1e-6), 0.0, 1.0)
    u8 = (x * 255.0).astype(np.uint8)
    rgb = np.stack([u8, u8, u8], axis=0)  # (3,H,W)
    
    return torch.from_numpy(rgb)

In [ ]:
def load_cellsam(ckpt_path: str, device: str) -> CellSAM:
    
    model = CellSAM({}).to(device).eval()
    sd = torch.load(os.path.expanduser(ckpt_path), map_location="cpu")
    model.load_state_dict(sd, strict=False)
    
    return model


def load_decode_head_into_cellsam(cellsam: CellSAM, new_ckpt_path: str):
    
    sd = torch.load(new_ckpt_path, map_location="cpu")
    if isinstance(sd, dict) and "state_dict" in sd:
        sd = sd["state_dict"]
        
    if any(k.startswith("cellfinder.decode_head.") for k in sd.keys()):
        stripped = {}
        for k, v in sd.items():
            if k.startswith("cellfinder.decode_head."):
                stripped[k[len("cellfinder.decode_head."):]] = v
        sd = stripped
    elif any(k.startswith("module.cellfinder.decode_head.") for k in sd.keys()):
        stripped = {}
        for k, v in sd.items():
            if k.startswith("module.cellfinder.decode_head."):
                stripped[k[len("module.cellfinder.decode_head."):]] = v
        sd = stripped

    missing, unexpected = cellsam.cellfinder.decode_head.load_state_dict(sd, strict=False)
    print(f"[INFO] Loaded new decode_head: missing={missing} unexpected={unexpected}")

# ---------- Inference ----------
@torch.no_grad()
def predict_boxes_cellsam(cellsam: CellSAM, img_raw: np.ndarray, score_thr: float):
    
    H, W = img_raw.shape
    img = mip_to_uint8_rgb(img_raw)  # CPU tensor uint8 (3,H,W)

    # CellSAM preprocessing
    x = cellsam.sam_bbox_preprocessing([img], percentile=False)  # (1,3,1024,1024) float on device

    # forward_inference returns boxes in the target_size space (here 1024x1024)
    res = cellsam.cellfinder.forward_inference(x)[0]
    boxes_1024 = res["boxes"].detach().cpu().numpy().astype(np.float32)   # (N,4) xyxy in 1024 pixels
    scores = res["scores"].detach().cpu().numpy().astype(np.float32)

    keep = scores >= score_thr
    boxes_1024 = boxes_1024[keep]
    scores = scores[keep]

    # scale to original image pixel space
    if len(boxes_1024) > 0:
        boxes = boxes_1024.copy()
        boxes[:, [0, 2]] *= (W / 1024.0)
        boxes[:, [1, 3]] *= (H / 1024.0)
    else:
        boxes = boxes_1024

    return boxes, scores

## Konfiguration

In [ ]:
import glob
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from skimage.measure import regionprops, find_contours

BASE      = "/home/hd/hd_hd/hd_tt333/new_ws/image_analysis/Voyage-r/Image Analysis/Mikroskop pipeline"
SPLIT_DIR = f"{BASE}/data/splits_noaug"          # Ordner mit train/val/test_bboxes.tsv
RAW_DIR   = f"{BASE}/data/01_raw"
MASK_DIR  = f"{BASE}/data/masks_gvs"
OUT_DIR   = f"{BASE}/results/test/cellfinder_v2"  # hier landen die PNGs (None = nicht speichern)

BASE_CKPT = "/home/hd/hd_hd/hd_tt333/.deepcell/models/cellsam_v1.2/cellsam_general.pt"
NEW_CKPT  = f"{BASE}/results/run_108_noaug/cellfinder/cellfinder_ft6_ckpt.pt"

SCORE_THR     = 0.4
BORDER_MARGIN = 1
SHOW_MASK     = True   # Umriss der annotierten Maske im Ground-Truth-Panel zeigen
device = "cuda" if torch.cuda.is_available() else "cpu"

## Modelle laden (nur einmal ausfuehren)

In [ ]:
old_cellsam = load_cellsam(BASE_CKPT, device=device)
new_cellsam = load_cellsam(BASE_CKPT, device=device)
load_decode_head_into_cellsam(new_cellsam, NEW_CKPT)

## Annotationen aus allen drei Splits laden

In [ ]:
parts = []
for split in ["train", "val", "test"]:
    p = os.path.join(SPLIT_DIR, f"{split}_bboxes.tsv")
    if os.path.exists(p):
        d = pd.read_csv(p, sep="\t")
        d["split"] = split
        parts.append(d)
ann = pd.concat(parts, ignore_index=True)
ann = ann[~ann["uid"].str.contains("_aug")]          # nur Originalbilder, keine Augmentierungen

# Uebersicht: ein Eintrag pro Bild
images = (ann.groupby("uid")
             .agg(split=("split", "first"), image_path=("image_path", "first"),
                  n_gt=("xmin", lambda s: int(s.notna().sum())))
             .reset_index())
print(images["split"].value_counts().to_string())
images

## Funktionen

In [ ]:
def to_numpy_boxes(b):
    if b is None:
        return np.zeros((0, 4), np.float32)
    if hasattr(b, "detach"):
        b = b.detach().cpu().numpy()
    return np.asarray(b, dtype=np.float32).reshape(-1, 4)

def draw_boxes(ax, boxes, color, lw=1.2):
    for x0, y0, x1, y1 in to_numpy_boxes(boxes):
        ax.add_patch(mpatches.Rectangle((x0, y0), x1 - x0, y1 - y0,
                                        fill=False, edgecolor=color, linewidth=lw))

def resolve_image(name):
    """uid (Dateiname ohne .tif) oder Pfad -> (uid, image_path)."""
    uid = os.path.splitext(os.path.basename(name))[0]
    row = images[images["uid"] == uid]
    if len(row):
        path = row["image_path"].iloc[0]
        # relative Pfade in den TSVs gelten ab dem Pipeline-Ordner, nicht ab dem Notebook
        return uid, path if os.path.isabs(path) else os.path.join(BASE, path)
    path = name if os.path.exists(name) else os.path.join(RAW_DIR, uid + ".tif")
    if not os.path.exists(path):
        raise FileNotFoundError(f"Bild nicht gefunden: {name}")
    return uid, path

def gt_boxes_for(uid):
    """GT-Boxen aus den TSVs; falls das Bild dort fehlt, direkt aus der Maske."""
    g = ann[ann["uid"] == uid]
    if len(g):
        return g[["xmin", "ymin", "xmax", "ymax"]].dropna().to_numpy(np.float32), g["split"].iloc[0]
    mask_path = os.path.join(MASK_DIR, f"{uid}_mask.tif")
    if os.path.exists(mask_path):
        b = [r.bbox for r in regionprops(tiff.imread(mask_path))]       # (y0, x0, y1, x1)
        b = np.array([[x0, y0, x1, y1] for y0, x0, y1, x1 in b], np.float32).reshape(-1, 4)
        return b, "nicht in Splits"
    return None, "keine Annotation"

def show_comparison(name, save=True):
    uid, img_path = resolve_image(name)
    img = read_image_any(img_path)
    H, W = img.shape

    gt, split = gt_boxes_for(uid)
    pred_old, _ = predict_boxes_cellsam(old_cellsam, img, score_thr=SCORE_THR)
    pred_new, _ = predict_boxes_cellsam(new_cellsam, img, score_thr=SCORE_THR)
    pred_old = filter_border_boxes(pred_old, H=H, W=W, margin=BORDER_MARGIN)
    pred_new = filter_border_boxes(pred_new, H=H, W=W, margin=BORDER_MARGIN)

    lo, hi = np.percentile(img, [1, 99])
    disp = np.clip((img.astype(np.float32) - lo) / (hi - lo + 1e-6), 0, 1)

    panels = [("Ground truth", gt, "lime"), ("CellSAM Basis", pred_old, "red"), ("Feingetunt", pred_new, "cyan")]
    fig, axes = plt.subplots(1, 3, figsize=(21, 7), sharex=True, sharey=True)
    for ax, (title, boxes, color) in zip(axes, panels):
        ax.imshow(disp, cmap="gray")
        if boxes is None:
            ax.set_title(f"{title}  (keine Annotation)")
        else:
            draw_boxes(ax, boxes, color)
            ax.set_title(f"{title}  (n={len(to_numpy_boxes(boxes))})")
        ax.axis("off")

    mask_path = os.path.join(MASK_DIR, f"{uid}_mask.tif")
    if SHOW_MASK and os.path.exists(mask_path):
        for c in find_contours(tiff.imread(mask_path) > 0, 0.5):
            axes[0].plot(c[:, 1], c[:, 0], color="yellow", lw=0.6)

    fig.suptitle(f"{uid}   [{split}]")
    plt.tight_layout()
    if save and OUT_DIR:
        os.makedirs(OUT_DIR, exist_ok=True)
        fig.savefig(os.path.join(OUT_DIR, f"compare_{uid}.png"), dpi=200)
    plt.show()

## Bilder auswaehlen und anzeigen

Eine der Varianten einkommentieren. Namen = Dateiname ohne `.tif` (siehe Tabelle oben).

In [ ]:
# a) bestimmte Bilder
auswahl = [
    "2026_07_XX V13 72h Expression_A1_3_CH4",
]

# b) alle Testbilder
# auswahl = images.loc[images["split"] == "test", "uid"].tolist()

# c) zufaellig 3 Bilder aus val
# auswahl = images[images["split"] == "val"].sample(3, random_state=0)["uid"].tolist()

# d) nur Bilder ohne GVs (Negativbeispiele) aus dem Test-Set
# auswahl = images[(images["split"] == "test") & (images["n_gt"] == 0)]["uid"].tolist()

for name in auswahl:
    show_comparison(name)